# Building A Co-citation Network

In [ ]:
# install and import libraries 
package_list <- c(
  "here", # use for paths creation
  "tidyverse",  # this package is essential 
  "janitor", # useful functions for cleaning imported data
  "biblionetwork", # creating edges
  "tidygraph", # for creating networks
  "ggraph" # plotting networks
  "ggdark"  # dark theme 
)
for (p in package_list) {
  if (p %in% installed.packages() == FALSE) {
    install.packages(p, dependencies = TRUE)
  }
  library(p, character.only = TRUE)
}
library(networkflow)
if (!require("BiocManager", quietly = TRUE))
    install.packages("BiocManager")
BiocManager::install(version = "3.15")
BiocManager::install("flowCore")
devtools::install_github("ParkerICI/vite")

In [ ]:
# read the dataset
data <- read_csv("/Users/Michael/Desktop/Core/inno1.csv", skip=1)
for (i in c(2:5)) {
    path <- "/Users/Michael/Desktop/Core/inno"
    file_path <- paste(path, as.character(i), ".csv", sep="")
    temp <- read_csv(file_path, skip=1)
    # combine the dataset and 
    data <- rbind(data, temp)
}

In [21]:
print(dim(data))  # 8942 times 15 

[1] 8942   15


In [24]:
print(names(data))

 [1] "Publication ID"                                         
 [2] "DOI"                                                    
 [3] "Title"                                                  
 [4] "Abstract"                                               
 [5] "Source title/Anthology title"                           
 [6] "PubYear"                                                
 [7] "Volume"                                                 
 [8] "Issue"                                                  
 [9] "Pagination"                                             
[10] "Authors"                                                
[11] "Authors Affiliations - Name of Research organization"   
[12] "Authors Affiliations - Country of Research organization"
[13] "Dimensions URL"                                         
[14] "Times cited"                                            
[15] "Cited references"                                       


In [25]:
# clean the column names (lower case with _)
data <- clean_names(data)
print(names(data))

 [1] "publication_id"                                       
 [2] "doi"                                                  
 [3] "title"                                                
 [4] "abstract"                                             
 [5] "source_title_anthology_title"                         
 [6] "pub_year"                                             
 [7] "volume"                                               
 [8] "issue"                                                
 [9] "pagination"                                           
[10] "authors"                                              
[11] "authors_affiliations_name_of_research_organization"   
[12] "authors_affiliations_country_of_research_organization"
[13] "dimensions_url"                                       
[14] "times_cited"                                          
[15] "cited_references"                                     


In [43]:
# rename columns with long name
# 11 is the index of authors_affiliations_name_of_research_organization
names(data)[11] <- "organization" 
names(data)[12] <- "country"
print(names(data))

 [1] "publication_id"               "doi"                         
 [3] "title"                        "abstract"                    
 [5] "source_title_anthology_title" "pub_year"                    
 [7] "volume"                       "issue"                       
 [9] "pagination"                   "authors"                     
[11] "organization"                 "country"                     
[13] "dimensions_url"               "times_cited"                 
[15] "cited_references"            


In [33]:
# check duplicates (many of them have a title 'Book Reviews')
print(dim(data[duplicated(data$title),]))  ## 114 


[1] 114  15


In [37]:
head(data[duplicated(data$title),][1:2, c(1, 2, 3)])

publication_id,doi,title
<chr>,<chr>,<chr>
pub.1148388295,10.1257/jel.60.2.636.r1,Book Reviews
pub.1138585739,10.1257/jel.59.2.651.r2,Book Reviews


In [39]:
# drop the duplicates
data <- data[!duplicated(data$title), ]
print(dim(data))  # 8942 (total articles) - 114 = 8828

[1] 8828   15


In [198]:
# create a table by mapping each author ot is affiliation
article_affiliation <- data %>%
                       select(publication_id, organization, country) %>%
                       separate_rows(c("organization","country"), sep=";")
head(article_affiliation)  # what a beautiful table (see publication_id)

publication_id,organization,country
<chr>,<chr>,<chr>
pub.1129556533,Bournemouth University,United Kingdom
pub.1129556533,University of Sarajevo,Bosnia and Herzegovina
pub.1129556533,University of Lincoln,United Kingdom
pub.1131662090,University of the Republic,Uruguay
pub.1132727712,Southern Methodist University,United States
pub.1132727712,New York University,United States


In [207]:
# define a function to remove the first space 
clean_strings <- function(strings) {
    str_len = nchar(strings)
    first_char <- substr(strings, 1, 1)
    if(identical(first_char, ' ')) {
        return(substr(strings, 2, str_len))
    } else {
        return(strings)
    }
}

In [208]:
clean_strings(" Massachusetts Institute of Technology")  # first space was removed

[1] "Massachusetts Institute of Technology"

In [201]:
clean_strings("United States")  # keep it same as there is no space

[1] "United States"

In [209]:
# you should try to use vectorize function
# however somehow it does not work in this case 
for (i in 1:nrow(article_affiliation)) {
    for (j in 1:ncol(article_affiliation)){
        article_affiliation[i, j] <- clean_strings(article_affiliation[i, j])
    }
}

In [203]:
print(dim(article_affiliation))  # it grows up to 13835 as we expand it  

[1] 13835     3


In [211]:
# get the top 10 publishing countries
top_publishing_country <- article_affiliation %>%
                    # drop NaN
                    filter(!is.na(country)) %>%
                    # count frequency of each observation's country
                    count(country) %>%
                    # calculate share 
                    mutate(country_share = n/sum(n)) %>%
                    # ranking based on country share 
                    slice_max(order_by = country_share, n = 10, 
                                                    with_ties = F)
top_publishing_country

country,n,country_share
<chr>,<int>,<dbl>
United States,2543,0.22932636
United Kingdom,1702,0.15348544
China,767,0.06916764
Netherlands,603,0.05437821
Germany,601,0.05419785
Italy,572,0.05158265
France,424,0.03823609
Spain,412,0.03715394
Canada,302,0.02723420


In [215]:
# get the top 10 publishing organizations 
top_publishing_org <- article_affiliation %>%
                    # drop NaN
                    filter(!is.na(organization)) %>%
                    # count frequency of each observation's country
                    count(organization) %>%
                    # calculate share 
                    mutate(organization_share = n/sum(n)) %>%
                    # ranking based on country share 
                    slice_max(order_by = organization_share, n = 10, 
                                                            with_ties = F)
top_publishing_org

organization,n,organization_share
<chr>,<int>,<dbl>
University of Sussex,157,0.014158175
University of Manchester,130,0.011723329
Massachusetts Institute of Technology,115,0.010370638
Copenhagen Business School,114,0.010280458
Harvard University,107,0.009649202
Utrecht University,97,0.008747407
Georgia Institute of Technology,91,0.008206331
University of Warwick,86,0.007755433
Erasmus University Rotterdam,77,0.006943818


In [314]:
# plot the data 
plot1 <- ggplot(top_publishing_country,aes(x=reorder(country, n),
                        y=n, fill=0))+coord_flip()+ geom_col()+ 
                        geom_text(aes(label = n), hjust = 1.3, 
                        color='white') + xlab('Country')+ 
                        ggtitle("Top 10 Publishing Countries")+
                        theme(legend.position = "none",
                        plot.title = element_text(hjust = 0.5),
                        axis.text.y = element_text(angle = 45))

In [315]:
plot2 <- ggplot(top_publishing_org,aes(x=reorder(organization, n), 
                        y=n, fill='red'))+coord_flip()+ geom_col()+ 
                        geom_text(aes(label = n), hjust = 1.3, 
                        color='black')+ xlab('Organizations')+ 
                        ggtitle("Top 10 Publishing Organizations")+ 
                        theme(legend.position = "none",
                        plot.title = element_text(hjust = 0.5),
                        axis.text.y = element_text(angle = 45))

In [ ]:
require(gridExtra)
png(file="toppub.png", width= 13, height= 6, units= "in", res= 900)
grid.arrange(plot1, plot2, ncol=2)
dev.off()

<img class="zoom-jupyter" src="/images/toppub.png">

In [305]:
# clean references
references <- data %>%
        filter(!is.na(cited_references)) %>%
        rename("citing_id"=publication_id) %>%
        select(citing_id, cited_references) %>%
        # match ; until [
        separate_rows(cited_references, sep=";(?=\\[)") %>%
        as_tibble
head(references)

citing_id,cited_references
<chr>,<chr>
pub.1129556533,"[Tödtling, Franz; Trippl, Michaela]|[ur.014146267347.37; ur.012365300247.26]|Research Policy|2005|34|8|1203-1219|10.1016/j.respol.2005.01.018|pub.1000672443|1146"
pub.1129556533,"[Rodríguez-Pose, Andrés]|[ur.015750100142.39]|Regional Studies|2013|47|7|1034-1047|10.1080/00343404.2012.748978|pub.1001930717|533"
pub.1129556533,"[Pippel, Gunnar; Seefeld, Vivian]|[ur.07733450565.36; ]|Economics of Innovation and New Technology|2015|25|5|455-469|10.1080/10438599.2015.1073480|pub.1002250442|17"
pub.1129556533,"[Coenen, Lars; Moodysson, Jerker; Martin, Hanna]|[ur.010344021157.27; ur.012213261362.53; ur.011042141073.58]|Regional Studies|2014|49|5|850-865|10.1080/00343404.2014.979321|pub.1003231776|89"
pub.1129556533,"[Vega-Jurado, Jaider; Gutiérrez-Gracia, Antonio; Fernández-de-Lucio, Ignacio; Manjarrés-Henríquez, Liney]|[ur.015036377177.19; ur.011106631311.73; ur.010523647377.01; ur.012535340130.06]|Research Policy|2008|37|4|616-632|10.1016/j.respol.2008.01.001|pub.1008010579|236"
pub.1129556533,"[Boschma, Ron]|[]|Regional Studies|2014|49|5|733-751|10.1080/00343404.2014.959481|pub.1009569019|522"


In [308]:
# separate references
column_names <- c("authors",
                  "author_id",
                  "source",
                  "year",
                  "volume",
                  "issue",
                  "pagination",
                  "doi",
                  "publication_id",
                  "times_cited")
direct_citation <- references %>%
                   separate(col=cited_references, into=column_names, sep="\\|")
head(direct_citation)
                    

Warning message:
“Expected 10 pieces. Additional pieces discarded in 10 rows [99323, 102318, 150066, 193591, 248892, 248899, 249090, 255419, 255423, 255428].”


citing_id,authors,author_id,source,year,volume,issue,pagination,doi,publication_id,times_cited
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
pub.1129556533,"[Tödtling, Franz; Trippl, Michaela]",[ur.014146267347.37; ur.012365300247.26],Research Policy,2005,34,8,1203-1219,10.1016/j.respol.2005.01.018,pub.1000672443,1146
pub.1129556533,"[Rodríguez-Pose, Andrés]",[ur.015750100142.39],Regional Studies,2013,47,7,1034-1047,10.1080/00343404.2012.748978,pub.1001930717,533
pub.1129556533,"[Pippel, Gunnar; Seefeld, Vivian]",[ur.07733450565.36; ],Economics of Innovation and New Technology,2015,25,5,455-469,10.1080/10438599.2015.1073480,pub.1002250442,17
pub.1129556533,"[Coenen, Lars; Moodysson, Jerker; Martin, Hanna]",[ur.010344021157.27; ur.012213261362.53; ur.011042141073.58],Regional Studies,2014,49,5,850-865,10.1080/00343404.2014.979321,pub.1003231776,89
pub.1129556533,"[Vega-Jurado, Jaider; Gutiérrez-Gracia, Antonio; Fernández-de-Lucio, Ignacio; Manjarrés-Henríquez, Liney]",[ur.015036377177.19; ur.011106631311.73; ur.010523647377.01; ur.012535340130.06],Research Policy,2008,37,4,616-632,10.1016/j.respol.2008.01.001,pub.1008010579,236
pub.1129556533,"[Boschma, Ron]",[],Regional Studies,2014,49,5,733-751,10.1080/00343404.2014.959481,pub.1009569019,522


In [318]:
references_network <- direct_citation %>%
                        # keep all distinct ids (as nodes)
                        distinct(publication_id, .keep_all = T) %>%
                        select(-citing_id)
tail(references_network)

authors,author_id,source,year,volume,issue,pagination,doi,publication_id,times_cited
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
"[Montgomery, David B.; Weinberg, Charles B.]",[; ],Journal of Marketing,1979,43,4,41,10.2307/1250269,pub.1069416093,68
"[Barksdale, Hiram C.; Darden, Bill]",[ur.07617707523.02; ur.011153410647.39],Journal of Marketing,1971,35,4,29,10.2307/1250454,pub.1069416257,95
"[McNamara, Carlton P.]",[ur.012317664362.64],Journal of Marketing,1972,36,1,50,10.2307/1250868,pub.1069416624,138
"[Zeithaml, Carl P.; Zeithaml, Valarie A.]",[ur.010535734725.59; ur.016623020210.20],Journal of Marketing,1984,48,2,46,10.2307/1251213,pub.1069416926,136
"[Webster, Frederick E.]",[],Journal of Marketing,1981,45,3,9,10.2307/1251538,pub.1069417199,119
"[Watkins, D. S.]",[],R and D Management,1973,3,2,65-70,10.1111/j.1467-9310.1973.tb01003.x,pub.1029469961,14


In [320]:
# building co-citation network
node_degree <- direct_citation %>%
                add_count(publication_id) %>%
                select(publication_id, n) %>%
                unique

filtered_references <- references_network %>%
                        left_join(node_degree) %>%
                        filter(n>=5)

edges <- biblionetwork::biblio_cocitation(filter(direct_citation,
                        publication_id %in% filtered_references$publication_id),
                        "citing_id",
                        "publication_id",
                        weight_threshold=3)


Joining, by = "publication_id"


In [322]:
head(edges)

from,to,weight,Source,Target
<chr>,<chr>,<dbl>,<chr>,<chr>
pub.1000006346,pub.1001300115,0.3340766,pub.1000006346,pub.1001300115
pub.1000006346,pub.1006722983,0.2760262,pub.1000006346,pub.1006722983
pub.1000006346,pub.1007138430,0.1463850,pub.1000006346,pub.1007138430
pub.1000006346,pub.1011091128,0.3273268,pub.1000006346,pub.1011091128
pub.1000006346,pub.1011303550,0.2519763,pub.1000006346,pub.1011303550
pub.1000006346,pub.1012726763,0.2592815,pub.1000006346,pub.1012726763


In [326]:
# make first column as node (publication_id)
filtered_references_reshape <- filtered_references %>%
                                relocate(publication_id, .before=authors)
cocitation_graph <- tbl_main_component(nodes=filtered_references_reshape,
                                       edges=edges,
                                       directed=FALSE)

In [327]:
cocitation_graph

# A tbl_graph: 10338 nodes and 149816 edges
#
# An undirected simple graph with 1 component
#
# Node Data: 10,338 × 11 (active)
  Id    authors author_id source year  volume issue pagination doi   times_cited
  <chr> <chr>   <chr>     <chr>  <chr> <chr>  <chr> <chr>      <chr> <chr>      
1 pub.… [Tödtl… [ur.0141… Resea… 2005  34     8     1203-1219  10.1… 1146       
2 pub.… [Rodrí… [ur.0157… Regio… 2013  47     7     1034-1047  10.1… 533        
3 pub.… [Vega-… [ur.0150… Resea… 2008  37     4     616-632    10.1… 236        
4 pub.… [Bosch… []        Regio… 2014  49     5     733-751    10.1… 522        
5 pub.… [Camag… [ur.0165… Growt… 2013  44     2     355-389    10.1… 222        
6 pub.… [Jense… [ur.0114… Resea… 2007  36     5     680-693    10.1… 887        
# … with 10,332 more rows, and 1 more variable: n <int>
#
# Edge Data: 149,816 × 5
   from    to weight Source         Target        
  <int> <int>  <dbl> <chr>          <chr>         
1  2303  4280  0.334 pub.1000006346 pub

In [ ]:
set.seed(1234)
graph <- leiden_workflow(cocitation_graph) # identifying clusters of nodes 

nb_communities <- graph %>% 
  activate(nodes) %>% 
  as_tibble %>% 
  select(Com_ID) %>% 
  unique %>% 
  nrow

# creating a color palette
palette <- scico::scico(n = nb_communities, palette = "hawaii") %>% 
           sample()
  
graph <- community_colors(graph, palette, community_column = "Com_ID")

graph <- graph %>% 
  activate(nodes) %>%
  mutate(size = n,# will be used for size of nodes
         first_author = str_extract(authors, "(?<=\\[)(.+?)(?=,)"),
         label = paste0(first_author, "-", year)) 

graph <- community_names(graph, 
                         ordering_column = "size", 
                         naming = "label", 
                         community_column = "Com_ID")

In [ ]:
# it takes around 17 minutes to train the model
graph <- vite::complete_forceatlas2(graph, 
                                    first.iter = 10000)

In [359]:
top_nodes  <- top_nodes(graph, 
                        ordering_column = "size", 
                        top_n = 15, 
                        top_n_per_com = 2,
                        biggest_community = TRUE,
                        community_threshold = 0.02)
                        
community_labels <- community_labels(graph, 
                                     community_name_column = "Community_name",
                                     community_size_column = "Size_com",
                                     biggest_community = TRUE,
                                     community_threshold = 0.02)

In [360]:
graph <- graph %>% 
  activate(edges) %>% 
  filter(weight > 0.05)

png(file="cocitation.png", width= 25, height= 25, units= "in", res= 900)
ggraph(graph, "manual", x = x, y = y) + 
  geom_edge_arc0(aes(color = color_edges, width = weight), alpha = 0.4,
  strength = 0.2, show.legend = FALSE) +
  scale_edge_width_continuous(range = c(0.1,2)) +
  scale_edge_colour_identity() +
  geom_node_point(aes(x=x, y=y, size = size, fill = color), pch = 21,
  alpha = 0.7, show.legend = FALSE) +
  scale_size_continuous(range = c(0.3,37)) +
  scale_fill_identity() +
  ggnewscale::new_scale("size") +
  ggrepel::geom_text_repel(data = top_nodes, aes(x=x, y=y, label = Label),
  size = 4, fontface="bold", alpha = 1,point.padding=NA,
  max.overlaps = 30, show.legend = FALSE) +
  ggrepel::geom_label_repel(data = community_labels, aes(x=x, y=y,
  label = Community_name, fill = color), size = 6, fontface="bold",
  alpha = 0.9, point.padding=NA, show.legend = FALSE) +
  scale_size_continuous(range = c(0.5,5)) +
  dark_theme_void()

dev.off()

Warning message:
“Existing variables `x`, `y` overwritten by layout variables”
